In [ ]:
import s3fs
import xarray as xr
import numpy as np

In [ ]:
rca_data_bucket = "ooi-data/" # contains data as it comes off the cabled array 
rca_advanced_qaqc_bucket = "rca-advanced-qaqc/" # contains data with additional value-added qaqc 

In [ ]:
fs = s3fs.S3FileSystem(anon=True)

In [ ]:
def load_data(stream_name, bucket):
    zarr_dir = bucket + stream_name
    zarr_store = fs.get_mapper(zarr_dir)
    ds = xr.open_zarr(zarr_store, consolidated=True)
    return ds

In [ ]:
ds = load_data("CE04OSPS-SF01B-2A-CTDPFA107-streamed-ctdpf_sbe43_sample", rca_data_bucket)

In [ ]:
ds

In [ ]:
ds.sea_water_temperature_qartod_results[:10].compute()

In [ ]:
ds.sea_water_temperature_qartod_results.flag_meanings

In [ ]:
np.unique(ds.sea_water_temperature_qartod_results.values)

In [ ]:
values, counts = np.unique(ds.sea_water_temperature_qartod_results.values, return_counts=True)
dict(zip(values, counts))

In [ ]:
import hvplot.xarray
import holoviews as hv
hv.extension("bokeh")

QARTOD_COLORS = {1: "blue", 2: "purple", 3: "yellow", 4: "red", 9: "orange"}
QARTOD_LABELS = {1: "pass", 2: "not_evaluated", 3: "suspect", 4: "fail", 9: "missing"}

# set to None to show all flags
SHOW_FLAGS = {2}

data = ds[["sea_water_temperature", "sea_water_temperature_qartod_results"]].sel(time=slice("2022-09", "2022-10")).compute()

line = data.sea_water_temperature.hvplot.line(
    x="time", color="lightgrey", line_width=1,
    width=900, height=400,
    ylabel="temperature (°C)", title="Sea Water Temperature with QARTOD Flags",
)

flags_to_plot = SHOW_FLAGS if SHOW_FLAGS is not None else QARTOD_COLORS.keys()

overlays = [line]
for flag_val in flags_to_plot:
    subset = data.sea_water_temperature.where(data.sea_water_temperature_qartod_results == flag_val, drop=True)
    if subset.size == 0:
        continue
    overlays.append(subset.hvplot.scatter(x="time", color=QARTOD_COLORS[flag_val], size=4, label=QARTOD_LABELS[flag_val]))

hv.Overlay(overlays).opts(legend_position="right")

In [ ]:
not_eval = ds.sea_water_temperature_qartod_results.sel(time=slice("2022", "2023")).where(
    ds.sea_water_temperature_qartod_results.sel(time=slice("2022", "2023")) == 2, drop=True
).compute()

not_eval.hvplot.scatter(
    x="time",
    y="sea_water_temperature_qartod_results",
    color="purple",
    size=2,
    title="not_evaluated (flag=2) occurrences 2022-2023 — sea_water_temperature",
    width=900,
    height=300,
)